In [1]:
import yaml
import pandas as pd
import mlflow
import datasets
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# GLOBAL VARIABLES
config_file = "../configs/run1_sentiment.yaml"

with open(config_file, "r") as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)
dataset_config = config.get("dataset")
model_config = config.get("model")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
experiment_name = config.get("mlflow", dict).get("experiment", None)
if experiment_name is None:
    raise ValueError(
        "Experiment name is not set in the config.\n"
        "Notebook does not run without tracking"
    )
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/13', creation_time=1791120060086, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1791120060086, lifecycle_stage='active', name='llm-bias-variance-2', tags={}, trace_location=None, workspace='default'>

In [5]:
dataset=datasets.load_dataset(dataset_config["name"])
model = AutoModelForCausalLM.from_pretrained(model_config["name"]).to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_config["name"], padding_side='left')
generation_config = GenerationConfig(**config.get("generation"))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 13246.16it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
train_batch_dataset = dataset["train"].batch(
    batch_size=32,
    drop_last_batch=False,
)
with torch.no_grad():
    for batch in train_batch_dataset:
        tokens = tokenizer(
            batch['review'], 
            return_tensors="pt", 
            padding=True
        ).to(model.device)
        output = model.generate(**tokens, generation_config=generation_config)
        break

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
tokenizer.decode(tokens['input_ids'][0][:530])

In [ ]:
tokens['input_ids']